<a href="https://colab.research.google.com/github/amanpoonia/SBeta_preparation/blob/main/Risk/VaR_CVaR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
VaR (Value at Risk): The potential loss in value of an asset/portfolio given:
                    1) A specific time period
                    2) A confidence level

CVaR (Conditional VaR): Also called as ES(Expected shortfall)
                      The average of losses beyond the VaR for a given confidence level.
'''

'''
This notebook contains:

1) VaR using 3 methods
1.a) Historical
1.b) Parametric
1.c) Simulation(Monte carlo)

2) CVaR using 3 methods
2.a) Historical
2.b) Parametric
2.c) Simulation
'''

In [65]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import datetime as dt
from scipy.stats import norm

# Download and close prices/returns data for a time period
years = 15
#start_date = dt.datetime.today() - dt.timedelta(days=365*years)
#end_date = dt.datetime.today()
start_date = '2008-04-28'
end_date = '2023-04-25'

tickers = ['SPY', 'BND', 'GLD', 'QQQ', 'VTI']
adj_close = pd.DataFrame()

for ticker in tickers:
  # Use multi_level_index=False to prevent MultiIndex columns and retrieve the adjusted 'Close'
  data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
  adj_close[ticker] = data['Close']

# Log Returns
log_ret = np.log(adj_close / adj_close.shift(1))
log_ret.dropna(inplace=True)

# Portfolio
# Weighing equally weighted
weights = np.array([1/len(tickers)]*len(tickers))

# Portfolio historical returns
p_ret = (log_ret * weights).sum(axis=1)


/tmp/ipykernel_3484/2486661206.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_3484/2486661206.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_3484/2486661206.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_3484/2486661206.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, multi_leve

In [ ]:
'''VaR Historical calculations'''
# Time period(days) and confidence level
n = 5
confidence_level = 0.95
Portfolio_value = 1000000

p_ndays_ret = p_ret.rolling(window=n).sum()
p_ndays_ret.dropna(inplace=True)
print(p_ndays_ret)

VaR = -np.percentile(p_ndays_ret, (1-confidence_level)*100)*Portfolio_value
print(VaR)

# visualisations

In [93]:
'''VaR Parametric calculations
The parametric method calculates VaR by assuming returns follow a normal distribution.
typically(normal but can be excess kurtosis and skewness)
and using mean and std dev of historical returns to estimate the potential losses at a given confidence level.
95%-single sided z score = 1.65
'''
n = 5
Portfolio_value = 1000000

p_ndays_ret = p_ret.rolling(window=n).sum()
p_ndays_ret.dropna(inplace=True)

# Need mean and std. dev. of portfolio for parametric
cov_matrix = log_ret.cov()*252
p_std_dev = np.sqrt(weights.T @ cov_matrix @ weights)


confidence_level = [0.90, 0.95, 0.99]
VaRs = []

for c in confidence_level:
  VaR = Portfolio_value *(p_ret.mean() * n - p_std_dev * norm.ppf(1-c) * np.sqrt(n/252))
  VaRs.append(VaR)

print(f'{'Confidence level':<20} {'Value at Risk':<20}')
print('-'*40)
for c in range(len(confidence_level)):
  print(f'{confidence_level[c]:.2f}{'':<15} ${VaRs[c]:.2f}')


Confidence level     Value at Risk       
----------------------------------------
0.90                $25652.48
0.95                $32479.35
0.99                $45285.41
